# Notebook 03 — Re-run the 10-seed extension (Part 2) of the final pipeline

Self-contained notebook that re-implements (in notebook cells) the extension of our final
pipeline (branch `use_llm_model`, folders `part1/` + `part2/`): download HDFS, rebuild the
Part 1 artifacts it needs, then **retrain and score all four models on 10 random splits
(seeds 0–9)** and rank them with Scott-Knott ESD — the resampling required by the assignment
(Extension, item 2).

| model | category | labels used | input |
|---|---|---|---|
| RF + MCV | classical, supervised | all | template counts per session |
| LSTM + BERT | deep learning, supervised | all | frozen BERT template embeddings, sliding windows 30/1 |
| PCA | classical, unsupervised | none | tf-idf weighted counts, SPE > Q-statistic |
| LogGPT | LLM-based, semi-supervised | normal-only filter | template-id sequences, GPT-2 next-key prediction |

Everything (data, caches, per-(model, seed) scores, ranking) lives on Google Drive under
`/content/drive/MyDrive/devops6309`, so any Colab session resumes where the previous one
stopped: finished (model, seed) pairs are skipped and LogGPT resumes from its checkpoint.

Runtime: **GPU runtime recommended** (Runtime → Change runtime type → T4 GPU). First run:
download ~2 min, parsing ~10–25 min (single-process Drain; their repo uses a 16-worker pool),
BERT embeddings ~1 min, RF+PCA seconds per seed, LSTM ~2–4 min per seed, LogGPT ~40–90 min
per seed (≈8–15 h for 10 seeds — run it across several Colab sessions, it resumes).

Sources ported cell by cell: `part1/src/{preprocess,drain,parse,partition,split,represent_mcv,
represent_bert}.py`, `part1/src/models/{rf,lstm}.py`, `part2/src/{scores.py,models/pca.py,
models/loggpt/*,run_resamples.py}`. Deviations are listed in the last cell.

In [ ]:
%pip install -q transformers pyarrow

In [ ]:
import json
import os
import shutil
import sys
import time
import urllib.request
import zipfile

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/devops6309'      # everything persists here
else:                                                # local fallback
    BASE = os.path.join(os.getcwd(), 'devops6309_cache')

DATA = os.path.join(BASE, 'data')                   # raw logs
CACHE = os.path.join(BASE, 'cache')                 # parsed / sessions / splits / features / models / loggpt
RESULTS = os.path.join(BASE, 'results')             # scores/, runs.csv, ranking
for d in (DATA, CACHE, RESULTS,
          os.path.join(CACHE, 'parsed'), os.path.join(CACHE, 'sessions'), os.path.join(CACHE, 'splits'),
          os.path.join(CACHE, 'features'), os.path.join(CACHE, 'models'), os.path.join(CACHE, 'loggpt'),
          os.path.join(RESULTS, 'scores')):
    os.makedirs(d, exist_ok=True)
os.environ['HF_HOME'] = os.path.join(BASE, 'hf_cache')   # keep BERT/GPT-2 downloads on Drive

SEEDS = list(range(10))            # the 10 resamples of the assignment's ranking requirement
MODELS = ['rf', 'lstm', 'pca', 'loggpt']

# part1/configs/hdfs.yaml + part2/configs/part2_hdfs.yaml (verified values)
CFG = dict(
    train_ratio=0.7,
    rf=dict(n_estimators=10, max_features='sqrt', max_depth=None, min_samples_split=2, bootstrap=True),
    window=dict(size=30, stride=1),
    lstm=dict(hidden=8, layers=1, dropout=0.0, lr=1e-3, epochs=30, batch_size=512,
              balanced_sampler=True, val_ratio=0.1),
    pca=dict(n_components=0.95, c_alpha=3.2905, term_weighting='tf-idf', normalization='zero-mean'),
    loggpt=dict(train_samples=5000, max_lens=512, n_layers=6, n_heads=6, n_embd=60, init_lr=1e-4,
                init_batch_size=16, init_num_epochs=100, logGPT_lr=1e-6, logGPT_episode=20,
                num_return_sequences=20, cut=[0.2], topk_ratio=0.5, score_batch_size=512),
)

import numpy as np
import pandas as pd
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('BASE =', BASE)
print('device =', DEVICE, '| torch', torch.__version__, '| transformers ready')
print('seeds:', SEEDS[0], '..', SEEDS[-1], '| models:', MODELS)

## 1 — Download HDFS (extension runs on HDFS only)

Loghub (Zenodo record 8196385), same URL as `part1/README.md`. Expected: 11,175,629 lines and
575,061 block labels (16,838 anomalous). Skipped when already on Drive.

In [ ]:
import glob

hdfs_log = os.path.join(DATA, 'HDFS.log')
label_csv = os.path.join(DATA, 'anomaly_label.csv')

if not (os.path.exists(hdfs_log) and os.path.exists(label_csv)):
    url = 'https://zenodo.org/api/records/8196385/files/HDFS_v1.zip/content'
    zip_path = os.path.join('/content', 'HDFS_v1.zip') if IN_COLAB else os.path.join(BASE, 'HDFS_v1.zip')
    if not os.path.exists(zip_path):
        print('downloading HDFS_v1.zip (~190 MB) ...', flush=True)
        t0 = time.time()
        urllib.request.urlretrieve(url, zip_path)
        print(f'  done in {time.time() - t0:.0f}s')
    tmp = os.path.join(os.path.dirname(zip_path), 'hdfs_extract')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmp)

    def find_one(pat):
        hits = [p for p in glob.glob(os.path.join(tmp, '**', pat), recursive=True) if os.path.isfile(p)]
        assert hits, pat
        return max(hits, key=os.path.getsize)
    shutil.copy2(find_one('HDFS*.log'), hdfs_log)
    shutil.copy2(find_one('anomaly_label*'), label_csv)
    shutil.rmtree(tmp, ignore_errors=True)

lab = pd.read_csv(label_csv)
n_anom = int(lab['Label'].eq('Anomaly').sum())
with open(hdfs_log, encoding='utf-8', errors='replace') as f:
    head = [next(f) for _ in range(2)]
print(''.join(head))
print(f'labels: {len(lab):,} blocks, {n_anom:,} anomalous ({n_anom / len(lab):.1%})')
print('  expected: 575,061 blocks, 16,838 anomalous (2.9%)')

## 2 — Parsing (port of `part1/src/preprocess.py` + `drain.py` + `parse.py`)

Qin et al. (SANER 2025) preprocessing framework, vendored verbatim in our repo: 14 ordered
regexes, all masks `"<*>"`, plus the **estimation stage** — the first 2,000 lines are masked
with the full sequence and only the regex types that matched there are applied to the rest.
Then logpai's Drain (depth 4, similarity 0.5, max 100 children). EventId = md5 of the final
template (logpai semantics), merged into integer TemplateId 1..n (0 = padding).

In [ ]:
# ---- part1/src/preprocess.py (verbatim) ----
import re

weekday_abb = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekday = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
month_abb = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August',
          'September', 'October', 'November', 'December']

QIN_REGEX = {
    'IPv4_port': r'(/|)(\d+\.){3}\d+(:\d+)?',
    'host_port': r'([\w-]+\.)+[\w-]+\:\d+',
    'package_host': r'([\w-]+\.){2,}[\w-]+(\$[\w-]+)*(\@[\w-]+)?',
    'Mac_address': r'^([0-9A-Fa-f]{2}[:-]){5}([0-9A-Fa-f]{2})$',
    'IPv6': r'(([0-9a-fA-F]{1,4}:){7}([0-9a-fA-F]{1,4}|:)|(([0-9a-fA-F]{1,4}:){1,7}|:):((:[0-9a-fA-F]{1,4}){1,7}|:))',
    'path': r'(/|)(([\w.-]+|\<\*\>)/)+([\w.-]+|\<\*\>)',
    'size': r'\b\d+\.?\d*\s?([KGTMkgtm]?(B|b)|([KGTMkgtm]))\b',
    'duration': r'\b\<?\d+\s?(sec|s|ms)\b',
    'block': r'blk\_\-?\d+',
    'numerical': r'\b(\-?\+?\d+\.?\d*)\b|\b0[Xx][a-fA-F\d]+\b|\b[a-fA-F\d]{4,}\b',
    'time': r'\b\d{2}:\d{2}(:\d{2}|:\d{2},\d+)?\b',
    'date': r'\b(\d{4}-\d{2}-\d{2})|\d{4}/\d{2}/\d{2}\b',
    'url': r'\bhttps?:\/\/(www\.)?[a-zA-Z0-9-]+(\.[a-zA-Z]{2,})+(:[0-9]{1,5})?(\/[^\s]*)?\b',
    'weekday_months': r'\b(%s)\b' % '|'.join(weekday_abb + weekday + month_abb + months),
}
QIN_SEQUENCE = ['url', 'IPv4_port', 'host_port', 'package_host', 'IPv6', 'Mac_address', 'time',
                'path', 'block', 'date', 'duration', 'size', 'numerical', 'weekday_months']
QIN_ESTIMATION_LINES = 2000
_QIN_COMPILED = {k: re.compile(v) for k, v in QIN_REGEX.items()}
_REPEAT_STAR = re.compile(r'(\<\*\>\s?)+\<\*\>')


def qin_preprocess(log, estimation_stage=False, use_sequence=QIN_SEQUENCE):
    matched_types = []
    new_log = log
    for key in use_sequence:
        new_log = _QIN_COMPILED[key].sub('<*>', new_log)
        if estimation_stage and new_log != log:
            matched_types.append(key)
            log = new_log
    new_log = _REPEAT_STAR.sub('<*>', new_log)
    if estimation_stage:
        return new_log, matched_types
    return new_log


class Masker:
    def __init__(self, mode, regexes=None, narrow=True):
        assert mode in ('none', 'regex', 'qin'), mode
        self.mode = mode
        self.narrow = narrow
        self.regexes = list(regexes or [])
        self._compiled = [re.compile(r) for r in self.regexes]
        self.qin_sequence = list(QIN_SEQUENCE)

    def estimate(self, first_contents):
        if self.mode != 'qin':
            return [self.mask(c) for c in first_contents]
        out, matched = [], set()
        for c in first_contents:
            m, types = qin_preprocess(c, estimation_stage=True)
            out.append(m)
            matched.update(types)
        if self.narrow:
            self.qin_sequence = [k for k in QIN_SEQUENCE if k in matched]
        return out

    def mask(self, content):
        if self.mode == 'none':
            return content
        if self.mode == 'regex':
            for r in self._compiled:
                content = r.sub('<*>', content)
            return content
        return qin_preprocess(content, estimation_stage=False, use_sequence=self.qin_sequence)


def logformat_regex(log_format):
    headers = []
    splitters = re.split(r'(<[^<>]+>)', log_format)
    regex = ''
    for k, s in enumerate(splitters):
        if k % 2 == 0:
            regex += re.sub(' +', r'\\s+', s)
        else:
            header = s.strip('<').strip('>')
            regex += '(?P<%s>.*?)' % header
            headers.append(header)
    return headers, re.compile('^' + regex + '$')


BLOCK_RE = re.compile(r'blk_-?\d+')


def hdfs_block_ids(content):
    return list(dict.fromkeys(BLOCK_RE.findall(content)))


LOG_FORMAT = '<Date> <Time> <Pid> <Level> <Component>: <Content>'   # part1/configs/hdfs.yaml

In [ ]:
# ---- part1/src/drain.py (verbatim; logpai Drain, streaming) ----
class Logcluster:
    __slots__ = ('logTemplate', 'idx', 'size')

    def __init__(self, logTemplate, idx):
        self.logTemplate = logTemplate
        self.idx = idx
        self.size = 0


class Node:
    __slots__ = ('childD', 'depth', 'digitOrtoken')

    def __init__(self, childD=None, depth=0, digitOrtoken=None):
        self.childD = dict() if childD is None else childD
        self.depth = depth
        self.digitOrtoken = digitOrtoken


class Drain:
    def __init__(self, depth=4, st=0.5, maxChild=100):
        self.depth = depth - 2
        self.st = st
        self.maxChild = maxChild
        self.rootNode = Node()
        self.clusters = []

    @staticmethod
    def hasNumbers(s):
        return any(char.isdigit() for char in s)

    def treeSearch(self, rn, seq):
        seqLen = len(seq)
        if seqLen not in rn.childD:
            return None
        parentn = rn.childD[seqLen]
        currentDepth = 1
        for token in seq:
            if currentDepth >= self.depth or currentDepth > seqLen:
                break
            if token in parentn.childD:
                parentn = parentn.childD[token]
            elif '<*>' in parentn.childD:
                parentn = parentn.childD['<*>']
            else:
                return None
            currentDepth += 1
        return self.fastMatch(parentn.childD, seq)

    def addSeqToPrefixTree(self, rn, logClust):
        seqLen = len(logClust.logTemplate)
        if seqLen not in rn.childD:
            firtLayerNode = Node(depth=1, digitOrtoken=seqLen)
            rn.childD[seqLen] = firtLayerNode
        else:
            firtLayerNode = rn.childD[seqLen]
        parentn = firtLayerNode
        currentDepth = 1
        for token in logClust.logTemplate:
            if currentDepth >= self.depth or currentDepth > seqLen:
                if len(parentn.childD) == 0:
                    parentn.childD = [logClust]
                else:
                    parentn.childD.append(logClust)
                break
            if token not in parentn.childD:
                if not self.hasNumbers(token):
                    if '<*>' in parentn.childD:
                        if len(parentn.childD) < self.maxChild:
                            newNode = Node(depth=currentDepth + 1, digitOrtoken=token)
                            parentn.childD[token] = newNode
                            parentn = newNode
                        else:
                            parentn = parentn.childD['<*>']
                    else:
                        if len(parentn.childD) + 1 < self.maxChild:
                            newNode = Node(depth=currentDepth + 1, digitOrtoken=token)
                            parentn.childD[token] = newNode
                            parentn = newNode
                        elif len(parentn.childD) + 1 == self.maxChild:
                            newNode = Node(depth=currentDepth + 1, digitOrtoken='<*>')
                            parentn.childD['<*>'] = newNode
                            parentn = newNode
                        else:
                            parentn = parentn.childD['<*>']
                else:
                    if '<*>' not in parentn.childD:
                        newNode = Node(depth=currentDepth + 1, digitOrtoken='<*>')
                        parentn.childD['<*>'] = newNode
                        parentn = newNode
                    else:
                        parentn = parentn.childD['<*>']
            else:
                parentn = parentn.childD[token]
            currentDepth += 1

    @staticmethod
    def seqDist(seq1, seq2):
        simTokens = 0
        numOfPar = 0
        for token1, token2 in zip(seq1, seq2):
            if token1 == '<*>':
                numOfPar += 1
                continue
            if token1 == token2:
                simTokens += 1
        return float(simTokens) / len(seq1), numOfPar

    def fastMatch(self, logClustL, seq):
        maxSim = -1
        maxNumOfPara = -1
        maxClust = None
        for logClust in logClustL:
            curSim, curNumOfPara = self.seqDist(logClust.logTemplate, seq)
            if curSim > maxSim or (curSim == maxSim and curNumOfPara > maxNumOfPara):
                maxSim = curSim
                maxNumOfPara = curNumOfPara
                maxClust = logClust
        if maxSim >= self.st:
            return maxClust
        return None

    @staticmethod
    def getTemplate(seq1, seq2):
        return [w if w == seq2[i] else '<*>' for i, w in enumerate(seq1)]

    def add(self, logmessageL):
        matchCluster = self.treeSearch(self.rootNode, logmessageL)
        if matchCluster is None:
            matchCluster = Logcluster(logTemplate=logmessageL, idx=len(self.clusters))
            self.clusters.append(matchCluster)
            self.addSeqToPrefixTree(self.rootNode, matchCluster)
        else:
            newTemplate = self.getTemplate(logmessageL, matchCluster.logTemplate)
            if newTemplate != matchCluster.logTemplate:
                matchCluster.logTemplate = newTemplate
        matchCluster.size += 1
        return matchCluster.idx

    def templates(self):
        return [' '.join(c.logTemplate) for c in self.clusters]

In [ ]:
# ---- part1/src/parse.py (HDFS branch; sequential instead of their 16-worker pool) ----
import hashlib
from array import array


def split_line(line, headers, regex):
    m = regex.search(line.strip())
    return {h: m.group(h) for h in headers} if m is not None else None


def finalize_templates(drain, cluster_per_line):
    # logpai semantics: EventId = md5(final template)[:8]; identical templates merge
    tmpl = drain.templates()
    eid = [hashlib.md5(t.encode('utf-8')).hexdigest()[0:8] for t in tmpl]
    first_seen = {}
    cluster_first = np.full(len(tmpl), np.iinfo(np.int64).max, dtype=np.int64)
    cl = np.asarray(cluster_per_line, dtype=np.int64)
    np.minimum.at(cluster_first, cl, np.arange(len(cl)))
    order = np.argsort(cluster_first, kind='stable')
    for c in order:
        if cluster_first[c] == np.iinfo(np.int64).max:
            continue
        first_seen.setdefault(eid[c], (len(first_seen) + 1, tmpl[c]))
    cluster_to_tid = np.zeros(len(tmpl), dtype=np.int32)
    for c in range(len(tmpl)):
        cluster_to_tid[c] = first_seen[eid[c]][0]
    tid = cluster_to_tid[cl]
    n = len(first_seen)
    occ = np.bincount(tid, minlength=n + 1)
    rows = sorted(((v[0], k, v[1]) for k, v in first_seen.items()))
    templates = pd.DataFrame(rows, columns=['TemplateId', 'EventId', 'EventTemplate'])
    templates['Occurrences'] = occ[templates.TemplateId.values]
    return tid.astype(np.int32), templates


def parse_hdfs(force=False):
    out = os.path.join(CACHE, 'parsed')
    npz_path, csv_path = os.path.join(out, 'lines.npz'), os.path.join(out, 'templates.csv')
    if os.path.exists(npz_path) and os.path.exists(csv_path) and not force:
        z = np.load(npz_path, allow_pickle=False)
        return z, pd.read_csv(csv_path, keep_default_na=False)
    headers, header_re = logformat_regex(LOG_FORMAT)
    masker = Masker('qin')
    drain = Drain(depth=4, st=0.5, maxChild=100)
    line_id, cluster = array('i'), array('i')
    pair_line, pair_block = array('i'), array('i')
    block_code = {}

    def add(lineno, cl_idx, f):
        row = len(line_id)
        line_id.append(lineno)
        cluster.append(cl_idx)
        for b in hdfs_block_ids(f['Content']):
            code = block_code.setdefault(b, len(block_code))
            pair_line.append(row)
            pair_block.append(code)

    t0, n_raw, n_unmatched = time.time(), 0, 0
    print(f'parsing {hdfs_log} (single process, ~10-25 min first run) ...', flush=True)
    with open(hdfs_log, encoding='utf-8', errors='replace') as fin:
        first_fields, first_lineno = [], []
        while len(first_fields) < QIN_ESTIMATION_LINES:
            line = fin.readline()
            if not line:
                break
            n_raw += 1
            f = split_line(line, headers, header_re)
            if f is not None:
                first_fields.append(f)
                first_lineno.append(n_raw)
        masked_first = masker.estimate([f['Content'] for f in first_fields])
        print('masking sequence after Qin estimation stage:', masker.qin_sequence)
        for lineno, f, m in zip(first_lineno, first_fields, masked_first):
            add(lineno, drain.add(m.strip().split()), f)
        for line in fin:
            n_raw += 1
            f = split_line(line, headers, header_re)
            if f is None:
                n_unmatched += 1
                continue
            add(n_raw, drain.add(masker.mask(f['Content']).strip().split()), f)
            if n_raw % 1_000_000 == 0:
                print(f'  {n_raw:,} lines, {len(drain.clusters)} clusters, {time.time() - t0:.0f}s', flush=True)

    tid, templates = finalize_templates(drain, cluster)
    templates.to_csv(csv_path, index=False)
    names = np.array(sorted(block_code, key=block_code.get))
    np.savez(npz_path,
             line_id=np.frombuffer(line_id, dtype=np.int32), template_id=tid,
             pair_line=np.frombuffer(pair_line, dtype=np.int32),
             pair_block=np.frombuffer(pair_block, dtype=np.int32), block_names=names)
    print(f'parsed {len(line_id):,}/{n_raw:,} lines ({n_unmatched} unmatched) -> '
          f'{len(templates)} templates in {time.time() - t0:.0f}s (team reference: 47)')
    return np.load(npz_path, allow_pickle=False), templates

parsed, TEMPLATES = parse_hdfs()
print(TEMPLATES.head(8).to_string(index=False))

## 3 — Sessions and splits (port of `part1/src/partition.py` + `split.py`)

HDFS: one session per block id, events in log order, labels from `anomaly_label.csv`
(unlabelled blocks dropped). Split: `numpy.random.default_rng(seed).permutation(n)`, first
70% train — the paper's random shuffle, **not** stratified, cached per seed. Expected:
575,061 sessions / 16,838 anomalous; HDFS seed 0 → 402,542 train / 172,519 test.

In [ ]:
from dataclasses import dataclass


@dataclass
class Sessions:
    offsets: np.ndarray   # int64 [n+1]
    events: np.ndarray    # int32 [total] template ids
    labels: np.ndarray    # int8 [n]
    keys: np.ndarray      # block ids

    def __len__(self):
        return len(self.labels)

    @property
    def lengths(self):
        return np.diff(self.offsets)

    def seq(self, i):
        return self.events[self.offsets[i]:self.offsets[i + 1]]

    def subset(self, idx):
        idx = np.asarray(idx, dtype=np.int64)
        lens = self.lengths[idx]
        new_off = np.zeros(len(idx) + 1, dtype=np.int64)
        np.cumsum(lens, out=new_off[1:])
        starts = self.offsets[idx]
        pos = np.repeat(starts - new_off[:-1], lens) + np.arange(new_off[-1])
        return Sessions(new_off, self.events[pos], self.labels[idx], self.keys[idx])

    def save(self, path):
        np.savez(path, offsets=self.offsets, events=self.events, labels=self.labels, keys=self.keys)

    @staticmethod
    def load(path):
        z = np.load(path, allow_pickle=False)
        return Sessions(z['offsets'], z['events'], z['labels'], z['keys'])


def _csr_from_groups(group, order_keys, tids):
    order = np.lexsort(tuple(order_keys[::-1]) + (group,)) if order_keys else np.argsort(group, kind='stable')
    g = group[order]
    n = int(g.max()) + 1 if len(g) else 0
    counts = np.bincount(g, minlength=n)
    offsets = np.zeros(n + 1, dtype=np.int64)
    np.cumsum(counts, out=offsets[1:])
    return offsets, tids[order].astype(np.int32)


def partition_hdfs():
    path = os.path.join(CACHE, 'sessions', 'sessions.npz')
    if os.path.exists(path):
        return Sessions.load(path)
    tid = parsed['template_id']
    pair_line, pair_block, names = parsed['pair_line'], parsed['pair_block'], parsed['block_names']
    # stable sort by block keeps file (time) order inside each block
    offsets, events = _csr_from_groups(pair_block.astype(np.int64), [], tid[pair_line])
    labels_df = pd.read_csv(label_csv)
    lab = dict(zip(labels_df.BlockId, (labels_df.Label == 'Anomaly').astype(np.int8)))
    keep = np.array([b in lab for b in names])
    labels = np.array([lab.get(b, -1) for b in names], dtype=np.int8)
    s = Sessions(offsets, events, labels, names)
    if not keep.all():
        print(f'{int((~keep).sum())} blocks without label dropped')
        s = s.subset(np.flatnonzero(keep))
    s.save(path)
    return s

SESSIONS = partition_hdfs()
print(f'sessions: {len(SESSIONS):,} ({int(SESSIONS.labels.sum()):,} anomalous, '
      f'{SESSIONS.labels.mean():.2%}) | events/session mean {SESSIONS.lengths.mean():.1f}')
print('  expected: 575,061 (16,838 anomalous, 2.93%)')


def make_split(labels, train_ratio, seed, method='random'):
    n = len(labels)
    if method == 'random':
        perm = np.random.default_rng(seed).permutation(n)
        n_train = int(n * train_ratio)
        train, test = np.sort(perm[:n_train]), np.sort(perm[n_train:])
    elif method == 'loglizer_uniform':
        train, test = [], []
        for c in (1, 0):
            idx = np.flatnonzero(labels == c)
            k = int(len(idx) * train_ratio)
            train.append(idx[:k])
            test.append(idx[k:])
        train, test = np.sort(np.concatenate(train)), np.sort(np.concatenate(test))
    else:
        raise ValueError(method)
    return train, test


def get_split(seed, force=False):
    path = os.path.join(CACHE, 'splits', f'hdfs_seed{seed}.json')
    if os.path.exists(path) and not force:
        d = json.load(open(path))
        return np.array(d['train']), np.array(d['test'])
    train, test = make_split(SESSIONS.labels, CFG['train_ratio'], seed)
    json.dump(dict(seed=seed, train=train.tolist(), test=test.tolist()), open(path, 'w'))
    return train, test

tr, te = get_split(SEEDS[0])
print(f'seed {SEEDS[0]}: train {len(tr):,} ({SESSIONS.labels[tr].mean():.2%} anomalous) | '
      f'test {len(te):,} ({SESSIONS.labels[te].mean():.2%} anomalous)')
print('  expected: 402,542 (2.94%) / 172,519 (2.91%)')

## 4 — Representations (port of `part1/src/represent_mcv.py` + `represent_bert.py`)

**MCV** (for RF and PCA): CSR count matrix, vocabulary = templates seen in train only.
**BERT embeddings** (for the LSTM): `bert-base-uncased`, mean of the second-to-last layer
over non-padding tokens (incl. [CLS]/[SEP]), max 50 tokens, frozen; one vector per template,
row 0 = padding. Cached on Drive (~440 MB HF download once).

In [ ]:
import scipy.sparse as sp


def count_matrix(sessions, n_cols):
    data = np.ones(len(sessions.events), dtype=np.float32)
    m = sp.csr_matrix((data, sessions.events.astype(np.int64), sessions.offsets),
                      shape=(len(sessions), n_cols))
    m.sum_duplicates()
    return m


class MCVVectorizer:
    def __init__(self, n_templates):
        self.n_cols = n_templates + 1
        self.vocab = None
        self.train_occurrences = None

    def fit(self, train_sessions):
        m = count_matrix(train_sessions, self.n_cols)
        occ = np.asarray(m.sum(axis=0)).ravel()
        self.vocab = np.flatnonzero(occ > 0)
        self.vocab = self.vocab[self.vocab != 0]
        self.train_occurrences = occ[self.vocab]
        return self

    def transform(self, sessions):
        return count_matrix(sessions, self.n_cols)[:, self.vocab].toarray().astype(np.float32)


def template_text(t, keep_placeholders=True):
    if keep_placeholders:
        return t
    s = re.sub(r'\s+', ' ', t.replace('<*>', ' ')).strip()
    return s or t


def compute_embeddings(templates, force=False):
    out = os.path.join(CACHE, 'features', 'bert_embeddings.npy')
    if os.path.exists(out) and not force:
        return np.load(out)
    from transformers import AutoModel, AutoTokenizer
    device = torch.device(DEVICE)
    tok = AutoTokenizer.from_pretrained('bert-base-uncased')
    model = AutoModel.from_pretrained('bert-base-uncased', output_hidden_states=True).to(device).eval()
    texts = [template_text(t, True) for t in templates.EventTemplate]
    E = np.zeros((len(templates) + 1, model.config.hidden_size), dtype=np.float32)
    rows = templates.TemplateId.values
    with torch.no_grad():
        for a in range(0, len(texts), 64):
            batch = texts[a:a + 64]
            enc = tok(batch, padding=True, truncation=True, max_length=50, return_tensors='pt').to(device)
            hs = model(**enc).hidden_states[-2]        # second-to-last layer, as the package
            m = enc['attention_mask'].unsqueeze(-1).to(hs.dtype)
            vec = (hs * m).sum(1) / m.sum(1)
            E[rows[a:a + len(batch)]] = vec.float().cpu().numpy()
    np.save(out, E)
    return E

E_BERT = compute_embeddings(TEMPLATES)
print('BERT embeddings:', E_BERT.shape, '(expected (48, 768): 47 templates + padding row)')

## 5 — Models

**RF** (port of `part1/src/models/rf.py`, replication-package settings: 10 trees, sqrt
features) and **LSTM** (port of `part1/src/models/lstm.py`: LSTM(768→8)→Linear(2), Adam
1e-3, 30 epochs, batch 512 windows, balanced session sampler, 10% validation split for
epoch selection; windows 30/1, session score = max window P(anomaly); packed sequences so
padding never reaches the last hidden state). The `package_faithful` ablation branches of
the original are not ported (Part 2 never uses them).

In [ ]:
from sklearn.ensemble import RandomForestClassifier


class RFMCV:
    name = 'RF'
    representation = 'MCV'

    def __init__(self, templates, params, seed=42):
        self.names = dict(zip(templates.TemplateId.astype(int), templates.EventTemplate))
        self.n_templates = len(templates)
        self.params = dict(params)
        self.seed = seed
        self.vectorizer = None
        self.columns = None
        self.clf = None

    def _frame(self, X, ids):
        return pd.DataFrame(X, columns=[self.names[int(i)] for i in ids])

    def fit(self, train_sessions):
        self.vectorizer = MCVVectorizer(self.n_templates).fit(train_sessions)
        X = self.vectorizer.transform(train_sessions)
        self.columns = np.asarray(self.vectorizer.vocab)
        self.clf = RandomForestClassifier(random_state=self.seed, n_jobs=-1, **self.params)
        # fit on a DataFrame so feature names are the template texts
        self.clf.fit(self._frame(X, self.columns), train_sessions.labels)
        return self

    def predict_scores(self, sessions):
        proba = self.clf.predict_proba(self._frame(self.vectorizer.transform(sessions), self.columns))
        classes = list(self.clf.classes_)
        return proba[:, classes.index(1)] if 1 in classes else np.zeros(len(sessions))

    @property
    def n_features(self):
        return len(self.columns)

In [ ]:
import copy
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.metrics import f1_score, precision_score, recall_score


def make_windows(sessions, win, stride):
    # -> W [N, win] int32, lengths [N], session index [N]; L <= w keeps the whole session,
    #    L > w slides 0, s, 2s... and appends the tail remainder as a shorter window
    lens = sessions.lengths
    sess, starts, wlen = [], [], []
    short = np.flatnonzero(lens <= win)
    sess.append(short)
    starts.append(np.zeros(len(short), dtype=np.int64))
    wlen.append(lens[short].astype(np.int64))
    for i in np.flatnonzero(lens > win):
        L = int(lens[i])
        st = np.arange(0, L - win + 1, stride, dtype=np.int64)
        ln = np.full(len(st), win, dtype=np.int64)
        if st[-1] + win < L:
            st = np.append(st, st[-1] + stride)
            ln = np.append(ln, L - st[-1])
        sess.append(np.full(len(st), i, dtype=np.int64))
        starts.append(st)
        wlen.append(ln)
    sess, starts, wlen = np.concatenate(sess), np.concatenate(starts), np.concatenate(wlen)
    order = np.lexsort((starts, sess))
    sess, starts, wlen = sess[order], starts[order], wlen[order]
    N = len(sess)
    W = np.zeros((N, win), dtype=np.int32)
    ar = np.arange(win)
    for a in range(0, N, 200_000):
        b = min(N, a + 200_000)
        idx = sessions.offsets[sess[a:b]][:, None] + starts[a:b][:, None] + ar[None, :]
        mask = ar[None, :] < wlen[a:b][:, None]
        W[a:b] = np.where(mask, sessions.events[np.minimum(idx, len(sessions.events) - 1)], 0)
    return W, wlen, sess


class LSTMNet(nn.Module):
    def __init__(self, embeddings, hidden, layers, dropout):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.from_numpy(embeddings), freeze=True, padding_idx=0)
        self.lstm = nn.LSTM(embeddings.shape[1], hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.fc = nn.Linear(hidden, 2)

    def forward(self, x, lengths):
        e = self.emb(x.long())
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.fc(h[-1])


class LSTMBert:
    name = 'LSTM'
    representation = 'BERT'

    def __init__(self, embeddings, window, params, seed=42):
        self.E = embeddings.astype(np.float32)
        self.win, self.stride = window['size'], window['stride']
        self.p = dict(params)
        self.seed = seed
        self.device = torch.device(DEVICE)
        self.net = None
        self.history = []
        self.best_epoch = None
        self.n_features = self.E.shape[1]

    def _seed(self):
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def _window_probs(self, W, L, bs=2048):
        self.net.eval()
        out = np.empty(len(W), dtype=np.float32)
        with torch.no_grad():
            for a in range(0, len(W), bs):
                x = torch.from_numpy(W[a:a + bs]).to(self.device)
                logits = self.net(x, torch.from_numpy(L[a:a + bs]))
                out[a:a + bs] = torch.softmax(logits, 1)[:, 1].float().cpu().numpy()
        return out

    @staticmethod
    def _session_max(win_probs, sess, n):
        s = np.zeros(n, dtype=np.float32)
        np.maximum.at(s, sess, win_probs)
        return s

    def fit(self, train_sessions):
        self._seed()
        rng = np.random.default_rng(self.seed)
        n = len(train_sessions)
        perm = rng.permutation(n)
        n_val = int(round(n * self.p['val_ratio']))
        val_idx, fit_idx = np.sort(perm[:n_val]), np.sort(perm[n_val:])
        fit_s, val_s = train_sessions.subset(fit_idx), train_sessions.subset(val_idx)
        Wf, Lf, Sf = make_windows(fit_s, self.win, self.stride)
        Wv, Lv, Sv = make_windows(val_s, self.win, self.stride)
        yf_sess = fit_s.labels.astype(np.int64)
        print(f'  LSTM: {len(fit_s):,} fit sessions ({int(yf_sess.sum()):,} anomalous) -> {len(Wf):,} windows | '
              f'{len(val_s):,} val sessions -> {len(Wv):,} windows', flush=True)
        self.net = LSTMNet(self.E, self.p['hidden'], self.p['layers'], self.p.get('dropout', 0.0)).to(self.device)
        opt = torch.optim.Adam([q for q in self.net.parameters() if q.requires_grad], lr=self.p['lr'])
        crit = nn.CrossEntropyLoss()
        Wf_t = torch.from_numpy(Wf).to(self.device)
        Lf_t = torch.from_numpy(Lf)
        yw_t = torch.from_numpy(yf_sess[Sf]).to(self.device)
        n_win_per_sess = np.bincount(Sf, minlength=len(fit_s))
        win_offsets = np.cumsum(np.concatenate([[0], n_win_per_sess]))
        if self.p.get('balanced_sampler', True):
            counts = np.bincount(yf_sess, minlength=2).astype(np.float64)
            w = 1.0 / counts[yf_sess]
            w /= w.sum()
        best = (-1.0, None)
        bs = self.p['batch_size']
        for ep in range(1, self.p['epochs'] + 1):
            t0 = time.time()
            self.net.train()
            if self.p.get('balanced_sampler', True):
                drawn = rng.choice(len(fit_s), size=len(fit_s), replace=True, p=w)
            else:
                drawn = rng.permutation(len(fit_s))
            reps = n_win_per_sess[drawn]
            widx = np.repeat(win_offsets[drawn], reps) \
                + (np.arange(reps.sum()) - np.repeat(np.cumsum(reps) - reps, reps))
            widx = widx[rng.permutation(len(widx))]
            tot, nb = 0.0, 0
            for a in range(0, len(widx), bs):
                ib_t = torch.from_numpy(widx[a:a + bs]).to(self.device)
                logits = self.net(Wf_t[ib_t], Lf_t[torch.from_numpy(widx[a:a + bs])])
                loss = crit(logits, yw_t[ib_t])
                opt.zero_grad()
                loss.backward()
                opt.step()
                tot += loss.item()
                nb += 1
            pv = self._session_max(self._window_probs(Wv, Lv), Sv, len(val_s))
            pred = (pv >= 0.5).astype(int)
            f1 = f1_score(val_s.labels, pred, zero_division=0)
            self.history.append(dict(epoch=ep, loss=tot / max(nb, 1), val_f1=f1, time_s=time.time() - t0))
            print(f'  epoch {ep:2d} loss {self.history[-1]["loss"]:.4f} | val F1 {f1:.3f} | '
                  f'{time.time() - t0:.1f}s', flush=True)
            if f1 > best[0]:
                best = (f1, copy.deepcopy(self.net.state_dict()))
                self.best_epoch = ep
        self.net.load_state_dict(best[1])
        print(f'  LSTM best epoch {self.best_epoch} (val F1 {best[0]:.3f})')
        return self

    def predict_scores(self, sessions):
        W, L, S = make_windows(sessions, self.win, self.stride)
        return self._session_max(self._window_probs(W, L), S, len(sessions))

**PCA** (port of `part2/src/models/pca.py`, loglizer MIT): tf-idf weighting + zero-mean on
the MCV, components covering 95% variance, score = SPE in the residual subspace, decision =
SPE > Q-statistic (alpha 0.001). Labels are never read (they are replaced by −1).

**LogGPT** (port of `part2/src/models/loggpt/`, CC-BY-NC, Nokia): GPT-2 (6 layers, 6 heads,
emb 60) trained from scratch on 5,000 normal sessions (4,500/500 train/val) to predict the
next template id, 100 epochs, then RL fine-tuning with a Top-K reward (+1 hit / −1 miss,
one policy-gradient update per generated sequence, early stop when validation stops
improving). Score = max rank of the actual next key (1 = most likely, `<unk>` = V+1);
anomalous ⟺ score > K, K = ⌊50% × unique training keys⌋. Checkpoints are saved every RL
episode so a Colab reset resumes mid-training.

In [ ]:
# ---- part2/src/scores.py (verbatim) ----
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

SCORE_DEF = {
    'rf': dict(label='RF + MCV', score='predict_proba[:, 1]', rule='>=', threshold='0.5'),
    'lstm': dict(label='LSTM + BERT', score='max window P(anomaly)', rule='>=', threshold='0.5'),
    'pca': dict(label='PCA', score='SPE (squared prediction error)', rule='>', threshold='Q-statistic'),
    'loggpt': dict(label='LogGPT', score='max rank of the actual next key', rule='>', threshold='K'),
    'loggpt_norl': dict(label='LogGPT w/o RL', score='max rank of the actual next key', rule='>', threshold='K'),
}
RANKED = ['rf', 'lstm', 'pca', 'loggpt']
LABELS_USED = {'rf': 'full', 'lstm': 'full', 'pca': 'none', 'loggpt': 'normal-only filter',
               'loggpt_norl': 'normal-only filter'}


def decide(model, scores, threshold):
    rule = SCORE_DEF[model]['rule']
    scores = np.asarray(scores)
    return (scores >= threshold if rule == '>=' else scores > threshold).astype(np.int8)


def session_metrics(y_true, scores, pred):
    y_true = np.asarray(y_true).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else float('nan')
    return dict(precision=float(p), recall=float(r), f1=float(f1), auc=float(auc),
                tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))


# ---- part2/src/models/pca.py (loglizer port) ----
class FeatureExtractor:
    def __init__(self, term_weighting='tf-idf', normalization='zero-mean'):
        self.term_weighting = term_weighting
        self.normalization = normalization
        self.idf_vec = None
        self.mean_vec = None

    def fit_transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        num_instance, num_event = X.shape
        if self.term_weighting == 'tf-idf':
            df_vec = np.sum(X > 0, axis=0)
            self.idf_vec = np.log(num_instance / (df_vec + 1e-8))
            X = X * self.idf_vec[None, :]
        if self.normalization == 'zero-mean':
            self.mean_vec = X.mean(axis=0).reshape(1, num_event)
            X = X - self.mean_vec
        return X

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.term_weighting == 'tf-idf':
            X = X * self.idf_vec[None, :]
        if self.normalization == 'zero-mean':
            X = X - self.mean_vec
        return X


class PCA:
    def __init__(self, n_components=0.95, threshold=None, c_alpha=3.2905):
        self.proj_C = None
        self.components = None
        self.n_components = n_components
        self.n_components_ = None
        self.threshold = threshold
        self.c_alpha = c_alpha
        self.explained = None

    def fit(self, X):
        num_instances, num_events = X.shape
        X_cov = np.dot(X.T, X) / float(num_instances)
        U, sigma, V = np.linalg.svd(X_cov)
        n_components = self.n_components
        if n_components < 1:
            total_variance = np.sum(sigma)
            variance = 0
            for i in range(num_events):
                variance += sigma[i]
                if variance / total_variance >= n_components:
                    break
            n_components = i + 1
        P = U[:, :n_components]
        I = np.identity(num_events, int)
        self.components = P
        self.proj_C = I - np.dot(P, P.T)
        self.n_components_ = int(n_components)
        self.explained = float(np.sum(sigma[:n_components]) / np.sum(sigma))
        if not self.threshold:
            # Q-statistic (Jackson & Mudholkar 1979), as in loglizer
            phi = np.zeros(3)
            for i in range(3):
                for j in range(n_components, num_events):
                    phi[i] += np.power(sigma[j], i + 1)
            h0 = 1.0 - 2 * phi[0] * phi[2] / (3.0 * phi[1] * phi[1])
            self.threshold = phi[0] * np.power(self.c_alpha * np.sqrt(2 * phi[1] * h0 * h0) / phi[0]
                                               + 1.0 + phi[1] * h0 * (h0 - 1) / (phi[0] * phi[0]),
                                               1.0 / h0)
        return self

    def spe(self, X):
        Y = X @ self.proj_C.T
        return np.einsum('ij,ij->i', Y, Y)

    def predict(self, X):
        return (self.spe(X) > self.threshold).astype(int)


class PCAMCV:
    name = 'PCA'

    def __init__(self, n_templates, params, seed=0):
        self.n_templates = n_templates
        self.p = dict(params)
        self.seed = seed
        self.vec = self.fe = self.pca = None

    def fit(self, train_sessions):
        self.vec = MCVVectorizer(self.n_templates).fit(train_sessions)
        self.fe = FeatureExtractor(self.p.get('term_weighting', 'tf-idf'),
                                   self.p.get('normalization', 'zero-mean'))
        X = self.fe.fit_transform(self.vec.transform(train_sessions))
        self.pca = PCA(self.p.get('n_components', 0.95), None, self.p.get('c_alpha', 3.2905)).fit(X)
        print(f'  PCA: {X.shape[1]} features, {self.pca.n_components_} components '
              f'({self.pca.explained:.4f} of variance), SPE threshold {self.pca.threshold:.4f}')
        return self

    @property
    def threshold(self):
        return float(self.pca.threshold)

    def info(self):
        return dict(n_features=int(len(self.vec.vocab)), n_components=self.pca.n_components_,
                    explained_variance=self.pca.explained, threshold=self.threshold)

    def predict_scores(self, sessions):
        return self.pca.spe(self.fe.transform(self.vec.transform(sessions)))

In [ ]:
# ---- part2/src/models/loggpt/vocab.py + model.py (port; CC-BY-NC-4.0, Nokia) ----
import math
from collections import Counter

from transformers import GPT2Config, GPT2LMHeadModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

SPECIALS = ['<pad>', '<unk>', '<eos>', '<bos>', '<mask>']


class WordVocab:
    pad_index, unk_index, eos_index, bos_index, mask_index = 0, 1, 2, 3, 4

    def __init__(self, texts):
        counter = Counter()
        for line in texts:
            for word in line:
                counter[word] += 1
        self.freqs = counter
        self.itos = list(SPECIALS)
        words_and_frequencies = sorted(counter.items(), key=lambda tup: tup[0])
        words_and_frequencies.sort(key=lambda tup: tup[1], reverse=True)
        for word, _ in words_and_frequencies:
            self.itos.append(word)
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    @property
    def n_keys(self):
        return len(self.itos) - len(SPECIALS)

    def forward(self, sentence):
        return [self.bos_index] + [self.stoi.get(word, self.unk_index) for word in sentence] + [self.eos_index]


def _pad_batch(seqs, pad, device):
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), pad, dtype=torch.long)
    m = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.as_tensor(s, dtype=torch.long)
        m[i, :len(s)] = 1
    return x.to(device), m.to(device)


class LogGPT:
    name = 'LogGPT'

    def __init__(self, params, seed=0, device='cuda', ckpt_dir=None):
        self.p = dict(params)
        self.seed = seed
        self.device = torch.device(device)
        self.ckpt_dir = ckpt_dir
        self.vocab = None
        self.K = None
        self.pre_model = None       # after pre-training (= "LogGPT w/o RL")
        self.ft_model = None        # after RL fine-tuning (= "LogGPT")
        self.pre_history, self.rl_history = [], []
        self.times = {}
        self.meta = {}

    def _info(self, msg, *a):
        print(('  ' + msg) % a if a else '  ' + msg, flush=True)

    def _seed(self):
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)
        torch.backends.cudnn.deterministic = True

    def _tokens(self, sessions, i):
        return [str(int(t)) for t in sessions.seq(i)]

    def _new_model(self):
        v = self.vocab
        config = GPT2Config(vocab_size=len(v), bos_token_id=v.bos_index, eos_token_id=v.eos_index,
                            pad_token_id=v.pad_index, unk_token_id=v.unk_index, mask_token_id=v.mask_index,
                            n_positions=self.p['max_lens'], output_hidden_states=True,
                            n_layer=self.p['n_layers'], n_head=self.p['n_heads'], n_embd=self.p['n_embd'])
        model = GPT2LMHeadModel(config)
        model.resize_token_embeddings(len(v))
        return model.to(self.device)

    def _ckpt(self, name):
        return None if self.ckpt_dir is None else os.path.join(self.ckpt_dir, name)

    def fit(self, normal_sessions):
        assert int(normal_sessions.labels.sum()) == 0, 'LogGPT must be trained on normal sessions only'
        seqs = [self._tokens(normal_sessions, i) for i in range(len(normal_sessions))]
        self.vocab = WordVocab(seqs)
        self.K = int(self.vocab.n_keys * self.p['topk_ratio'])
        self.meta.update(n_train_sessions=len(seqs), n_unique_train_keys=self.vocab.n_keys,
                         vocab_size=len(self.vocab), K=self.K,
                         max_train_len=max(len(s) for s in seqs))
        self._info('LogGPT: %d normal sessions, %d unique keys -> vocab %d, K = %d',
                   len(seqs), self.vocab.n_keys, len(self.vocab), self.K)
        enc = [self.vocab.forward(s) for s in seqs]
        assert max(len(e) for e in enc) <= self.p['max_lens']
        n_fit = int(0.9 * len(enc))                    # original: valid_index = int(0.9 * len(train_df))
        self.fit_enc, self.val_enc = enc[:n_fit], enc[n_fit:]
        t0 = time.time()
        self._pretrain()
        self.times['pretrain_s'] = time.time() - t0 + self.times.get('pretrain_resumed_s', 0.0)
        t0 = time.time()
        self._rl_finetune()
        self.times['rl_s'] = time.time() - t0 + self.times.get('rl_resumed_s', 0.0)
        return self

    def _pretrain(self):
        path = self._ckpt('pretrain.pt')
        if path is not None and os.path.exists(path):
            ck = torch.load(path, map_location=self.device, weights_only=False)
            assert ck['itos'] == self.vocab.itos
            self.pre_model = self._new_model()
            self.pre_model.load_state_dict(ck['last_state'])
            self.pre_history, self.times['pretrain_resumed_s'] = ck['history'], ck['time_s']
            self._info('LogGPT pre-training cache hit: %s', path)
            return
        self._seed()
        model = self._new_model()
        optim = torch.optim.AdamW(model.parameters(), lr=self.p['init_lr'])
        bs, pad = self.p['init_batch_size'], self.vocab.pad_index
        best_loss, best_epoch = math.inf, None
        t_start = time.time()
        for epoch in range(self.p['init_num_epochs']):
            t0 = time.time()
            model.train()
            tr_loss, nb = 0.0, 0
            for a in range(0, len(self.fit_enc), bs):     # DataLoader(shuffle=False) in the original
                x, m = _pad_batch(self.fit_enc[a:a + bs], pad, self.device)
                optim.zero_grad()
                loss = model(x, attention_mask=m, labels=x).loss
                loss.backward()
                optim.step()
                tr_loss += loss.item()
                nb += 1
            model.eval()
            val_loss, nv = 0.0, 0
            with torch.no_grad():
                for a in range(0, len(self.val_enc), bs):
                    x, m = _pad_batch(self.val_enc[a:a + bs], pad, self.device)
                    val_loss += model(x, attention_mask=m, labels=x).loss.item()
                    nv += 1
            val_loss /= nv
            if val_loss < best_loss:
                best_loss, best_epoch = val_loss, epoch
            self.pre_history.append(dict(epoch=epoch, train_loss=tr_loss / nb, val_loss=val_loss,
                                         time_s=time.time() - t0))
            if epoch % 10 == 0 or epoch == self.p['init_num_epochs'] - 1:
                self._info('  pre-train epoch %3d  train loss %.4f  val loss %.4f  (%.1fs)',
                           epoch, tr_loss / nb, val_loss, time.time() - t0)
        # the original keeps the LAST-epoch weights in memory (never reloads the best checkpoint)
        self.pre_model = model
        self.meta['pretrain_best_val_epoch'] = best_epoch
        self._info('LogGPT pre-training done in %.1fs; best val loss at epoch %d (%.4f), last epoch used',
                   time.time() - t_start, best_epoch, best_loss)
        if path is not None:
            torch.save(dict(last_state=model.state_dict(), best_epoch=best_epoch,
                            history=self.pre_history, itos=self.vocab.itos,
                            time_s=time.time() - t_start), path)

    def _prompt(self, seq_ids):
        lens = len(seq_ids)
        i = self.p['cut'][0]
        if math.floor(i * lens) < 2:
            return seq_ids[:2]
        if math.floor(i * lens) >= lens - 2:
            return seq_ids[:-2]
        return seq_ids[:math.floor(i * lens)]

    def _generate(self, model, seq_ids):
        lens = len(seq_ids)
        input_ids = torch.tensor([self._prompt(seq_ids)], device=self.device)
        out = model.generate(input_ids, max_length=lens, min_length=lens,
                             num_beams=self.p['num_return_sequences'], do_sample=True,
                             pad_token_id=self.vocab.pad_index, num_return_sequences=1,
                             early_stopping=False, return_dict_in_generate=False,
                             output_hidden_states=False)
        assert out.shape[1] == lens, (out.shape, lens)
        return out

    def _rl_loss(self, model, seq_ids, generated):
        # -(sum_t log p(actual k_t | generated prefix)) x (sum_t +-1 Top-K reward)
        logits = model(generated).logits[0, :-1][1:]      # [lens-2, V]
        ys = torch.tensor(seq_ids[2:], device=logits.device)
        log_prob = torch.log_softmax(logits.float(), dim=-1).gather(1, ys[:, None]).sum()
        topk = logits.topk(self.K, dim=-1).indices
        hits = (topk == ys[:, None]).any(dim=1)
        reward = int(hits.sum().item()) * 2 - len(ys)     # +1 per hit, -1 per miss
        return -log_prob * reward

    def _rl_step(self, model, optim, seq_ids):
        model.train()
        optim.zero_grad()
        generated = self._generate(model, seq_ids)
        loss = self._rl_loss(model, seq_ids, generated)
        loss.backward()
        optim.step()

    def _valid_step(self, model, seq_ids):
        model.eval()
        with torch.no_grad():
            generated = self._generate(model, seq_ids)
            return self._rl_loss(model, seq_ids, generated).item()

    def _val_loss(self, model):
        return sum(self._valid_step(model, s) for s in self.val_enc) / len(self.val_enc)

    def _rl_finetune(self):
        path = self._ckpt('rl_state.pt')
        ft = copy.deepcopy(self.pre_model)                # original: copy.deepcopy(initGPT_model)
        optim = torch.optim.Adam(ft.parameters(), lr=self.p['logGPT_lr'])
        eps = 1e-7
        if path is not None and os.path.exists(path):
            st = torch.load(path, map_location='cpu', weights_only=False)   # RNG states stay on CPU
            ft.load_state_dict(st['ft_state'])
            optim.load_state_dict(st['optim_state'])
            torch.set_rng_state(st['torch_rng'])
            torch.cuda.set_rng_state(st['cuda_rng'])
            rs = np.random.RandomState()
            rs.set_state(st['np_state'])
            best_state, best_loss, last_loss, count = st['best_state'], st['best_loss'], st['last_loss'], st['count']
            start, self.rl_history, done = st['next_episode'], st['history'], st['done']
            self.times['rl_resumed_s'] = st['time_s']
            self._info('LogGPT RL: resuming at episode %d (done=%s)', start, done)
        else:
            self._seed()
            rs = np.random.RandomState(self.seed)
            t0 = time.time()
            init_loss = self._val_loss(ft)
            best_loss = last_loss = init_loss
            best_state, count, start, done = None, 0, 0, False
            self.rl_history = [dict(episode=-1, val_loss=init_loss, time_s=time.time() - t0)]
            self._info('LogGPT RL: initial val loss %.4f (%.1fs)', init_loss, time.time() - t0)
        t_run = time.time()
        prior = self.times.get('rl_resumed_s', 0.0)
        for episode in range(start, self.p['logGPT_episode']):
            if done:
                break
            t0 = time.time()
            random_seed = rs.randint(0, 100)              # original: np.random.seed(np.random.randint(0, 100))
            rs.seed(random_seed)
            order = rs.permutation(len(self.fit_enc))
            for j in order:
                self._rl_step(ft, optim, self.fit_enc[j])
            t_train = time.time() - t0
            cur = self._val_loss(ft)
            saved = False
            if cur <= best_loss or episode == 0:
                best_loss = cur
                best_state = copy.deepcopy(ft.state_dict())
                saved = True
            rel = (last_loss - cur) / (cur + eps)
            if rel < 0.01:
                count += 1
            last_loss = cur
            self.rl_history.append(dict(episode=episode, val_loss=cur, rel_improvement=rel,
                                        count=count, saved=saved, train_s=t_train,
                                        time_s=time.time() - t0))
            self._info('  RL episode %2d  val loss %.4f  rel.impr. %+.4f  count %d  %s (%.1fs)',
                       episode, cur, rel, count, 'saved' if saved else '', time.time() - t0)
            if count > 3:
                self._info('  RL early stop after episode %d', episode)
                done = True
            if path is not None:
                torch.save(dict(ft_state=ft.state_dict(), optim_state=optim.state_dict(),
                                best_state=best_state, best_loss=best_loss, last_loss=last_loss,
                                count=count, next_episode=episode + 1, done=done,
                                history=self.rl_history, torch_rng=torch.get_rng_state(),
                                cuda_rng=torch.cuda.get_rng_state(), np_state=rs.get_state(),
                                time_s=prior + time.time() - t_run), path)
        self.ft_model = self._new_model()
        self.ft_model.load_state_dict(best_state)         # original test(): self.FT_GPT._load('FT')
        self.meta['rl_episodes_run'] = sum(1 for h in self.rl_history if h['episode'] >= 0)

    @torch.no_grad()
    def max_rank_scores(self, sessions, which='ft', batch_size=512, unk_not_in_topk=True):
        # Max rank of the actual next key over all predicted positions; distinct sequences
        # are scored once; <unk> target gets rank V + 1 (their brief rule)
        model = self.ft_model if which == 'ft' else self.pre_model
        model.eval()
        V = len(self.vocab)
        groups, inv = {}, np.empty(len(sessions), dtype=np.int64)
        for i in range(len(sessions)):
            inv[i] = groups.setdefault(sessions.seq(i).tobytes(), len(groups))
        uniq = [None] * len(groups)
        for i in range(len(sessions)):
            if uniq[inv[i]] is None:
                uniq[inv[i]] = self.vocab.forward(self._tokens(sessions, i))
        order = np.argsort([len(u) for u in uniq], kind='stable')
        uscore = np.zeros(len(uniq), dtype=np.int64)
        for a in range(0, len(order), batch_size):
            ib = order[a:a + batch_size]
            x, m = _pad_batch([uniq[k] for k in ib], self.vocab.pad_index, self.device)
            logits = model(x, attention_mask=m).logits[:, :-1][:, 1:].float()   # [B, L-2, V]
            probs = torch.softmax(logits, dim=-1)
            tgt = x[:, 2:]
            p_t = probs.gather(-1, tgt[..., None])
            rank = 1 + (probs > p_t).sum(-1)
            if unk_not_in_topk:
                rank = torch.where(tgt == self.vocab.unk_index, torch.full_like(rank, V + 1), rank)
            lens = m.sum(1)
            valid = torch.arange(tgt.shape[1], device=self.device)[None, :] < (lens - 2)[:, None]
            rank = torch.where(valid, rank, torch.zeros_like(rank))
            uscore[ib] = rank.max(1).values.cpu().numpy()
        return uscore[inv]

    @torch.no_grad()
    def repo_predict_topk(self, sessions, which='ft', n_limit=50):
        # Verbatim logic of the original InitGPT._predict_topk on the first n_limit sessions
        # (spot-check only; the full verification lives in part2/src/run_resamples.py)
        model = self.ft_model if which == 'ft' else self.pre_model
        model.eval()
        y_pred = []
        for i in range(min(n_limit, len(sessions))):
            seq_ids = self.vocab.forward(self._tokens(sessions, i))
            lst_ys = seq_ids[2:]
            input_ids = torch.tensor([seq_ids]).to(self.device)
            logits = model(input_ids).logits[:, :-1, :][:, 1:, :]
            probabilities = torch.softmax(logits, dim=-1)
            topk = torch.topk(probabilities, k=self.K)
            lst_topk = topk.indices.tolist()[0]
            lst_hit = [1 if val in lst_topk[ind] else 0 for ind, val in enumerate(lst_ys)]
            y_pred.append(1 if sum(lst_hit) < len(lst_ys) else 0)
        return np.array(y_pred)

## 6 — The 10-seed resampling loop (port of `part2/src/run_resamples.py`)

For each seed 0–9: draw the split with Part 1's function, retrain every model, score the
**full test split** (all models of a seed see identical test sessions — asserted), write
`scores/{model}_seed{seed}.parquet` + a JSON sidecar, and rebuild `runs.csv`. Finished
(model, seed) pairs are skipped, so the notebook can be re-run across Colab sessions.

LogGPT's training sessions: 5,000 normal sessions of the train split, sampled with the
resample seed (labels used only for this filter), cached per seed. Expected timing per seed:
RF ~1 s, PCA ~1 s, LSTM ~2–4 min, LogGPT ~40–90 min.

In [ ]:
RUN_COLS = ['seed', 'model', 'n_train_sessions', 'labels_used', 'precision', 'recall', 'f1', 'auc',
            'tp', 'fp', 'tn', 'fn', 'train_time_s', 'infer_time_s', 'device', 'threshold']


def unlabeled(sessions):
    # copy whose labels are all -1: proves that an unsupervised model never reads them
    return Sessions(sessions.offsets, sessions.events,
                    np.full(len(sessions), -1, dtype=np.int8), sessions.keys)


class Ctx:
    def __init__(self, seed):
        self.seed = seed
        self.templates = TEMPLATES
        self.tr, self.te = get_split(seed)
        self.train_s, self.test_s = SESSIONS.subset(self.tr), SESSIONS.subset(self.te)
        self.test_ids = SESSIONS.keys[self.te]
        print(f'seed {seed}: train {len(self.tr):,} ({int(self.train_s.labels.sum()):,} anomalous) | '
              f'test {len(self.te):,} ({int(self.test_s.labels.sum()):,} anomalous)')

    def normal_sample(self):
        path = os.path.join(CACHE, 'loggpt', f'seed{self.seed}', 'train_sessions.json')
        n = CFG['loggpt']['train_samples']
        if os.path.exists(path):
            d = json.load(open(path))
            assert d['n'] == n and d['n_train_split'] == len(self.tr)
            return np.array(d['session_idx'])
        os.makedirs(os.path.dirname(path), exist_ok=True)
        normal_tr = self.tr[SESSIONS.labels[self.tr] == 0]
        pick = np.random.default_rng(self.seed).choice(normal_tr, size=n, replace=False)
        json.dump(dict(seed=self.seed, n=n, n_train_split=len(self.tr),
                       n_normal_in_train_split=len(normal_tr), session_idx=pick.tolist()),
                  open(path, 'w'))
        return pick


def write_result(ctx, r):
    name = r['model']
    scores = np.asarray(r['scores'], dtype=np.float64)
    pred = decide(name, scores, r['threshold'])
    assert len(scores) == len(ctx.te)
    df = pd.DataFrame(dict(session_idx=ctx.te.astype(np.int64), session_id=ctx.test_ids.astype(str),
                           label=ctx.test_s.labels.astype(np.int8), score=scores, prediction=pred))
    df.to_parquet(os.path.join(RESULTS, 'scores', f'{name}_seed{ctx.seed}.parquet'), index=False)
    met = session_metrics(df.label.values, scores, pred)
    json.dump(dict(seed=ctx.seed, model=name, n_train_sessions=r['n_train_sessions'],
                   labels_used=LABELS_USED[name], train_time_s=r['train_time_s'],
                   infer_time_s=r['infer_time_s'], device=r['device'], n_test=len(df),
                   n_test_anomalous=int(df.label.sum()), threshold=float(r['threshold']),
                   ranked=name in RANKED, metrics=met, info=r.get('info')),
              open(os.path.join(RESULTS, 'scores', f'{name}_seed{ctx.seed}.json'), 'w'))
    print(f'[{name}] seed {ctx.seed}: P {met["precision"]:.4f} R {met["recall"]:.4f} '
          f'F1 {met["f1"]:.4f} AUC {met["auc"]:.4f} | train {r["train_time_s"]:.1f}s', flush=True)


def run_rf(ctx):
    model = RFMCV(ctx.templates, CFG['rf'], ctx.seed)
    t0 = time.time(); model.fit(ctx.train_s); t_train = time.time() - t0
    t0 = time.time(); scores = model.predict_scores(ctx.test_s); t_inf = time.time() - t0
    return [dict(model='rf', scores=scores, threshold=0.5, train_time_s=t_train, infer_time_s=t_inf,
                 device='cpu', n_train_sessions=len(ctx.train_s),
                 info=dict(n_features=int(model.n_features)))]


def run_lstm(ctx):
    model = LSTMBert(E_BERT, CFG['window'], CFG['lstm'], ctx.seed)
    t0 = time.time(); model.fit(ctx.train_s); t_train = time.time() - t0
    t0 = time.time(); scores = model.predict_scores(ctx.test_s); t_inf = time.time() - t0
    return [dict(model='lstm', scores=scores, threshold=0.5, train_time_s=t_train, infer_time_s=t_inf,
                 device=DEVICE, n_train_sessions=len(ctx.train_s),
                 info=dict(best_epoch=model.best_epoch))]


def run_pca(ctx):
    model = PCAMCV(len(ctx.templates), CFG['pca'], ctx.seed)
    t0 = time.time(); model.fit(unlabeled(ctx.train_s)); t_train = time.time() - t0
    t0 = time.time(); scores = model.predict_scores(ctx.test_s); t_inf = time.time() - t0
    return [dict(model='pca', scores=scores, threshold=model.threshold, train_time_s=t_train,
                 infer_time_s=t_inf, device='cpu', n_train_sessions=len(ctx.train_s),
                 info=model.info())]


def run_loggpt(ctx):
    pick = ctx.normal_sample()
    normal = SESSIONS.subset(pick)
    assert int(normal.labels.sum()) == 0 and np.isin(pick, ctx.tr).all() and not np.isin(pick, ctx.te).any()
    ckpt = os.path.join(CACHE, 'loggpt', f'seed{ctx.seed}')
    os.makedirs(ckpt, exist_ok=True)
    model = LogGPT(CFG['loggpt'], ctx.seed, 'cuda' if torch.cuda.is_available() else 'cpu', ckpt)
    model.fit(normal)
    out = []
    for which, name, t_train in (('pre', 'loggpt_norl', model.times['pretrain_s']),
                                 ('ft', 'loggpt', model.times['pretrain_s'] + model.times['rl_s'])):
        t0 = time.time()
        scores = model.max_rank_scores(ctx.test_s, which, CFG['loggpt']['score_batch_size'])
        t_inf = time.time() - t0
        repo = model.repo_predict_topk(ctx.test_s, which)
        ours = (scores[:len(repo)] > model.K).astype(int)
        print(f'[{name}] spot-check vs original logic on {len(repo)} sessions: '
              f'{int((ours != repo).sum())} mismatches')
        out.append(dict(model=name, scores=scores, threshold=model.K, train_time_s=t_train,
                        infer_time_s=t_inf, device=DEVICE, n_train_sessions=len(normal),
                        info=dict(times=model.times, **model.meta)))
    return out


RUNNERS = {'rf': run_rf, 'lstm': run_lstm, 'pca': run_pca, 'loggpt': run_loggpt}
OUTPUTS = {'loggpt': ['loggpt', 'loggpt_norl']}


def rebuild_runs():
    rows = []
    for f in sorted(glob.glob(os.path.join(RESULTS, 'scores', '*_seed*.json'))):
        m = json.load(open(f))
        rows.append({**{k: m[k] for k in RUN_COLS if k in m}, **m['metrics']})
    if not rows:
        return None
    df = pd.DataFrame(rows)[RUN_COLS]
    order = {k: i for i, k in enumerate(RANKED + ['loggpt_norl'])}
    df = df.sort_values(['seed', 'model'], key=lambda s: s.map(order) if s.name == 'model' else s)
    df = df.reset_index(drop=True)
    df.to_csv(os.path.join(RESULTS, 'runs.csv'), index=False)
    return df


# ---- the loop: seeds 0-9 x models; finished (model, seed) pairs are skipped ----
import glob

for seed in SEEDS:
    for m in MODELS:
        outs = OUTPUTS.get(m, [m])
        done = all(os.path.exists(os.path.join(RESULTS, 'scores', f'{o}_seed{seed}.parquet')) and
                   os.path.exists(os.path.join(RESULTS, 'scores', f'{o}_seed{seed}.json')) for o in outs)
        if done:
            print(f'[{m}] seed {seed}: cached, skipped')
            continue
        print(f'=== [{m}] seed {seed}: training ===', flush=True)
        ctx = Ctx(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        for r in RUNNERS[m](ctx):
            write_result(ctx, r)
    # fairness check: identical test sessions across the models of this seed
    files = sorted(glob.glob(os.path.join(RESULTS, 'scores', f'*_seed{seed}.parquet')))
    if files:
        ids0 = pd.read_parquet(files[0], columns=['session_id']).session_id.values
        assert all((pd.read_parquet(f, columns=['session_id']).session_id.values == ids0).all()
                   for f in files), f'test set mismatch at seed {seed}'
        print(f'seed {seed}: {len(files)} score files on identical test sessions '
              f'({len(ids0):,})', flush=True)

RUNS = rebuild_runs()
RUNS

## 7 — Summary and Scott-Knott ESD ranking

Reference medians from the team's published run (`part2/results/RESULTS_PART2.md`): RF .998,
LSTM .979, LogGPT .845, PCA .769 (F1). The ranking uses the same R package as the team
(`ScottKnottESD` ≥ 3.0.0, non-parametric, alpha 0.05, Cliff's δ ≥ 0.147 merge) — installed
from r-universe (~2–4 min, once per Colab session).

In [ ]:
if RUNS is not None:
    agg = (RUNS[RUNS.model.isin(RANKED)]
           .groupby('model')[['precision', 'recall', 'f1', 'auc']]
           .agg(['median', 'mean', 'std']).round(4))
    display(agg)
    display(RUNS.groupby('model')[['train_time_s', 'infer_time_s']].median().round(2))

In [ ]:
%%writefile sk_rank.R
#!/usr/bin/env Rscript
# Port of part2/src/scott_knott.R (Scott-Knott ESD, np version) over runs.csv
suppressPackageStartupMessages({
  library(ScottKnottESD)
  library(effsize)
})
stopifnot(packageVersion('ScottKnottESD') >= '3.0.0')
res_dir <- commandArgs(trailingOnly = TRUE)[1]
ranked <- c(rf = 'RF_MCV', lstm = 'LSTM_BERT', pca = 'PCA', loggpt = 'LogGPT')
runs <- read.csv(file.path(res_dir, 'runs.csv'), stringsAsFactors = FALSE)
runs <- runs[runs$model %in% names(ranked), ]
for (metric in c('f1', 'auc')) {
  wide <- reshape(runs[, c('seed', 'model', metric)], idvar = 'seed', timevar = 'model', direction = 'wide')
  wide <- wide[order(wide$seed), ]
  x <- data.frame(row.names = wide$seed, check.names = FALSE)
  for (m in names(ranked)) x[[ranked[[m]]]] <- wide[[paste0(metric, '.', m)]]
  # drop models that do not have all seeds yet (e.g. only loggpt seed 0 finished)
  incomplete <- names(x)[sapply(x, function(col) anyNA(col))]
  if (length(incomplete)) {
    cat(sprintf('NOTE: %s skipped in this ranking (missing seeds)\n', paste(incomplete, collapse = ', ')))
    x <- x[!names(x) %in% incomplete]
  }
  stopifnot(!anyNA(x), nrow(x) >= 2, ncol(x) >= 2)
  write.csv(cbind(seed = wide$seed, x), file.path(res_dir, sprintf('sk_input_%s.csv', metric)),
            row.names = FALSE)
  sk <- sk_esd(x, version = 'np', alpha = 0.05)
  groups <- sk$groups
  out <- data.frame(model = names(groups), group = as.integer(groups),
                    median = sapply(names(groups), function(m) median(x[[m]])),
                    mean = sapply(names(groups), function(m) mean(x[[m]])),
                    n_resamples = nrow(x), row.names = NULL)
  write.csv(out, file.path(res_dir, sprintf('sk_%s.csv', metric)), row.names = FALSE)
  cat(sprintf('== %s (1 = best group) ==\n', toupper(metric)))
  print(out)
}

In [ ]:
!Rscript -e "install.packages('effsize', repos='https://cloud.r-project.org', quiet=TRUE); install.packages('ScottKnottESD', repos=c('https://klainfo.r-universe.dev','https://cloud.r-project.org'), quiet=TRUE)"
!Rscript sk_rank.R "{RESULTS}"

In [ ]:
import glob
from IPython.display import display

for f in sorted(glob.glob(os.path.join(RESULTS, 'sk_*.csv'))):
    if 'input' in f or 'effect' in f:
        continue
    print('---', os.path.basename(f))
    display(pd.read_csv(f))
print('\nsaved to Drive:', RESULTS)
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if d not in ('hf_cache',)]
    for f in files:
        p = os.path.join(root, f)
        print(f'  {os.path.getsize(p) / 1e6:9.2f} MB  {os.path.relpath(p, BASE)}')

## Notes and deviations vs `part1/` + `part2/`

- **Same**: algorithms, hyper-parameters, seeds, window/split/score/threshold rules, the
  fairness protocol (identical test sessions per seed, labels only where allowed), the
  per-(model, seed) caching, LogGPT checkpoint resume, and the R Scott-Knott ESD ranking.
- **Different**: the parsing loop is single-process here (their 16-worker pool — Colab has
  2 vCPUs), ~10–25 min for HDFS once; Python is Colab's 3.12 stack instead of their pinned
  3.13 venv; `structured.parquet` and the date/time arrays are not written (nothing
  downstream reads them); the LSTM's `package_faithful` ablation branches, `pca_normal5k`
  sensitivity run, the full LogGPT verification harness, SHAP (RQ4) and `report.py` are not
  ported — the team's `part2/src` has them.
- **Resume**: re-running the notebook skips finished (model, seed) pairs; LogGPT resumes an
  interrupted RL fine-tuning from `cache/loggpt/seed<k>/rl_state.pt`. Run LogGPT seeds
  across several Colab sessions if needed (MODELS = ['rf', 'lstm', 'pca'] first, then
  MODELS = ['loggpt']).